<h3>Preprocessing, BoW, TF-IDF, Logistic Regression, train/test split, confusion matrix, unseen prediction, and comparison</h3>
<h3>Author: Vicky Law</h3>
<h3>Apr 17, 2026</h3>

# Step 1: Import Libraries

In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 2: Load dataset

In [3]:
# Read the CSV file into a pandas DataFrame
df = pd.read_csv("text-processing2.csv")

# Display the first 5 rows
print("First 5 rows of the dataset:")
print(df.head())

# Check dataset shape
print("\nDataset shape:")
print(df.shape)

First 5 rows of the dataset:
   ID                                               Text  Label
0   1      User logged in successfully from known device      0
1   2  Employee accessed internal system during worki...      0
2   3  File uploaded to company server by authorized ...      0
3   4            Regular backup completed without errors      0
4   5        User browsed company website and logged out      0

Dataset shape:
(20, 3)


# Step 3: Explore dataset

In [4]:
# Show column names
print("\nColumn names:")
print(df.columns)

# Check data types and missing values
print("\nDataset info:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

# Check class distribution
print("\nLabel distribution:")
print(df["Label"].value_counts())

# View some normal and attack samples
print("\nSample normal logs:")
print(df[df["Label"] == 0]["Text"].head())

print("\nSample attack logs:")
print(df[df["Label"] == 1]["Text"].head())


Column names:
Index(['ID', 'Text', 'Label'], dtype='object')

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      20 non-null     int64 
 1   Text    20 non-null     object
 2   Label   20 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 612.0+ bytes
None

Missing values:
ID       0
Text     0
Label    0
dtype: int64

Label distribution:
Label
0    10
1    10
Name: count, dtype: int64

Sample normal logs:
0        User logged in successfully from known device
1    Employee accessed internal system during worki...
2    File uploaded to company server by authorized ...
3              Regular backup completed without errors
4          User browsed company website and logged out
Name: Text, dtype: object

Sample attack logs:
10    Multiple failed login attempts from unknown IP...
11    User clicked suspicious email link requestin

### Observation:
The dataset contains 20 rows and 3 columns: ID, Text, and Label.
There are no missing values, and the labels are balanced with 10 normal logs and 10 attack logs.

# Step 4: Text preprocessing

In [5]:
# Define a function to preprocess text
def preprocess_text(text):
    """
    This function:
    1. Converts text to lowercase
    2. Tokenizes words using regex
    3. Removes stopwords (optional but used here)
    4. Joins tokens back into a cleaned string
    """
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Tokenize: keep only alphabetic words
    tokens = re.findall(r'\b[a-z]+\b', text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    
    # Return cleaned text
    return " ".join(tokens)

# Apply preprocessing to the Text column
df["Clean_Text"] = df["Text"].apply(preprocess_text)

# Show original and cleaned text
print("\nOriginal vs Cleaned text:")
print(df[["Text", "Clean_Text"]].head())


Original vs Cleaned text:
                                                Text  \
0      User logged in successfully from known device   
1  Employee accessed internal system during worki...   
2  File uploaded to company server by authorized ...   
3            Regular backup completed without errors   
4        User browsed company website and logged out   

                                     Clean_Text  
0         user logged successfully known device  
1      employee accessed internal working hours  
2  file uploaded company server authorized user  
3               regular backup completed errors  
4           user browsed company website logged  


### Observation:
The text was successfully converted to lowercase, tokenized, and cleaned by removing stopwords.
This helps standardize the logs before feature extraction.

# Step 5: Define features and split the data

In [6]:
# X = input text, y = target label
X = df["Clean_Text"]
y = df["Label"]

# Split data into training set (80%) and testing set (20%)
# stratify=y keeps the class balance consistent
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining size:", len(X_train))
print("Testing size:", len(X_test))


Training size: 16
Testing size: 4


# Step 6: Build Bag of Words features

In [7]:
# Create a CountVectorizer object
# This converts text into word count vectors
bow_vectorizer = CountVectorizer()

# Fit on training data and transform training text into numeric vectors
X_train_bow = bow_vectorizer.fit_transform(X_train)

# Transform test data using the same vocabulary
X_test_bow = bow_vectorizer.transform(X_test)

print("\nBoW training feature shape:", X_train_bow.shape)
print("BoW testing feature shape:", X_test_bow.shape)


BoW training feature shape: (16, 72)
BoW testing feature shape: (4, 72)


# Step 7: Train and evaluate Logistic Regression with BoW

In [8]:
# Create Logistic Regression model
bow_model = LogisticRegression(max_iter=1000)

# Train the model
bow_model.fit(X_train_bow, y_train)

# Predict on test data
y_pred_bow = bow_model.predict(X_test_bow)

# Evaluate the model
bow_accuracy = accuracy_score(y_test, y_pred_bow)
bow_cm = confusion_matrix(y_test, y_pred_bow)

print("\n=== Bag of Words Model Results ===")
print("Accuracy:", bow_accuracy)
print("Confusion Matrix:")
print(bow_cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bow))


=== Bag of Words Model Results ===
Accuracy: 0.5
Confusion Matrix:
[[1 1]
 [1 1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4



### Observation:
The Bag of Words model achieved 0.50 accuracy on the test set.
The confusion matrix shows that the model correctly classified 1 normal log and 1 attack log,
but also misclassified 2 logs.

# Step 8: Build TF-IDF features

In [9]:
# Create a TF-IDF vectorizer
# This gives higher weight to important words
# and lower weight to very common words
tfidf_vectorizer = TfidfVectorizer()

# Fit on training data and transform training text
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("\nTF-IDF training feature shape:", X_train_tfidf.shape)
print("TF-IDF testing feature shape:", X_test_tfidf.shape)


TF-IDF training feature shape: (16, 72)
TF-IDF testing feature shape: (4, 72)


# Step 9: Train and evaluate Logistic Regression with TF-IDF

In [10]:
# Create Logistic Regression model
tfidf_model = LogisticRegression(max_iter=1000)

# Train the model
tfidf_model.fit(X_train_tfidf, y_train)

# Predict on test data
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

# Evaluate the model
tfidf_accuracy = accuracy_score(y_test, y_pred_tfidf)
tfidf_cm = confusion_matrix(y_test, y_pred_tfidf)

print("\n=== TF-IDF Model Results ===")
print("Accuracy:", tfidf_accuracy)
print("Confusion Matrix:")
print(tfidf_cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf))


=== TF-IDF Model Results ===
Accuracy: 0.5
Confusion Matrix:
[[1 1]
 [1 1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4



### Observation:
The TF-IDF model also achieved 0.50 accuracy on the test set.
This indicates that, on this small dataset, TF-IDF did not outperform Bag of Words.

# Step 10: Find highest TF-IDF words for attack logs

In [11]:
# Select only attack logs
attack_texts = df[df["Label"] == 1]["Clean_Text"]

# Fit TF-IDF only on attack logs
attack_tfidf_vectorizer = TfidfVectorizer()
attack_tfidf_matrix = attack_tfidf_vectorizer.fit_transform(attack_texts)

# Take the mean TF-IDF value of each column (each word) across all attack logs
# axis=0 means calculate the average by column
# .A1 converts the result from a matrix format into a 1D NumPy array
feature_names = attack_tfidf_vectorizer.get_feature_names_out()
avg_scores = attack_tfidf_matrix.mean(axis=0).A1

# Create a DataFrame for easy sorting
tfidf_scores_df = pd.DataFrame({
    "Word": feature_names,
    "Average_TFIDF_Score": avg_scores
})

# Sort from highest to lowest
tfidf_scores_df = tfidf_scores_df.sort_values(by="Average_TFIDF_Score", ascending=False)

print("\nTop 10 highest TF-IDF words in attack logs:")
print(tfidf_scores_df.head(10))


Top 10 highest TF-IDF words in attack logs:
           Word  Average_TFIDF_Score
21         file             0.075855
45   suspicious             0.075345
16        email             0.074143
50         user             0.072420
10  credentials             0.072420
30        login             0.070181
27     infected             0.050000
17   encrypting             0.050000
22        files             0.050000
38   ransomware             0.050000


### Observation:
The top TF-IDF words for attack logs include file, suspicious, email, credentials, login, and ransomware. These words are strongly related to malicious or suspicious activities and help distinguish attack logs from normal logs.

# Step 11: Predict unseen inputs

In [17]:
# New unseen log examples
unseen_logs = [
    "User logged into internal portal successfully",
    "Suspicious email attachment executed malicious code",
    "Multiple failed login attempts detected from external IP",
    "Regular backup completed and system health is normal"
]

# Preprocess unseen logs
clean_unseen_logs = [preprocess_text(text) for text in unseen_logs]

# Transform using the trained TF-IDF vectorizer
unseen_vectors = tfidf_vectorizer.transform(clean_unseen_logs)

# Predict using TF-IDF model
unseen_predictions = tfidf_model.predict(unseen_vectors)

print("\n=============== Unseen Input Predictions ===============\n")
for log, pred in zip(unseen_logs, unseen_predictions):
    label_name = "Attack" if pred == 1 else "Normal"
    print(f"Log: {log}")
    print(f"Prediction: {label_name}")
    print("-" * 60)


=============== Unseen Input Predictions ===============

Log: User logged into internal portal successfully
Prediction: Normal
------------------------------------------------------------
Log: Suspicious email attachment executed malicious code
Prediction: Attack
------------------------------------------------------------
Log: Multiple failed login attempts detected from external IP
Prediction: Attack
------------------------------------------------------------
Log: Regular backup completed and system health is normal
Prediction: Attack
------------------------------------------------------------


### Observation:
The model correctly identified some suspicious unseen logs as attacks, but it also classified one normal-looking log as an attack. This suggests that the model can detect attack-related language, but its reliability is limited by the small dataset size and limited training examples.

# Step 12: Compare BoW and TF-IDF performance

In [18]:
print("\n=== Model Comparison ===")
print(f"BoW Accuracy: {bow_accuracy}")
print(f"TF-IDF Accuracy: {tfidf_accuracy}")

if tfidf_accuracy > bow_accuracy:
    print("TF-IDF performed better than BoW.")
elif bow_accuracy > tfidf_accuracy:
    print("BoW performed better than TF-IDF.")
else:
    print("BoW and TF-IDF performed the same on this test split.")


=== Model Comparison ===
BoW Accuracy: 0.5
TF-IDF Accuracy: 0.5
BoW and TF-IDF performed the same on this test split.


### Observation:
Both Bag of Words and TF-IDF achieved the same accuracy of 0.50 on this test split. This suggests that neither representation had a clear advantage on this small dataset. <br>

# Step 13: Reflection

One limitation of this assignment is the small dataset size of only 20 logs. Because of this, the model may not learn enough patterns to generalize well to new data. A larger and more diverse cybersecurity log dataset would likely improve performance.

### 1. Compare Bag of Words vs TF-IDF performance
Bag of Words and TF-IDF both achieved an accuracy of 0.50 on the test set in this assignment. This means that, for this particular dataset and split, neither method performed better than the other. Bag of Words represents text based only on raw word frequency, while TF-IDF gives more importance to distinctive words and reduces the weight of words that appear frequently across documents. In theory, TF-IDF often performs better because it highlights more meaningful terms. However, in this assignment, the dataset is very small, containing only 20 logs, so TF-IDF did not show a clear advantage over Bag of Words. With a larger and more diverse dataset, TF-IDF would likely provide better discrimination between attack and normal logs.

### 2. How could this model help a real cybersecurity system?
This model could help a real cybersecurity system by automatically classifying system or security logs as normal activity or possible attack activity. It could support security analysts by quickly identifying suspicious events such as malicious email actions, failed login attempts, ransomware-related behavior, or unusual system activity. In a real environment, this type of model could be integrated into a security monitoring pipeline to assist with early threat detection, reduce manual analysis time, and improve response speed. Although the model in this assignment is simple and trained on a very small dataset, it demonstrates how text analytics and machine learning can be applied to cybersecurity for faster and more automated threat identification.

### 3. Which words have the highest TF-IDF scores for attack logs?
The words with the highest TF-IDF scores in the attack logs include file, suspicious, email, credentials, login, and ransomware. These words are strongly associated with malicious behavior and help distinguish attack logs from normal logs.

## Conclusion:
In conclusion, this assignment demonstrated how text preprocessing and machine learning can be used to classify cybersecurity logs as normal or attack activity. Both Bag of Words and TF-IDF achieved the same performance on this small dataset, showing that dataset size is a key factor in model effectiveness. Despite its limitations, this approach shows the potential of text analytics to support automated cybersecurity monitoring and threat detection.